In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.colors as mcolorsx
from matplotlib.patches import Circle

from regions import Regions

import os
from glob import glob

from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.wcs.utils import pixel_to_skycoord, skycoord_to_pixel
from astropy.nddata import Cutout2D
import astropy.units as u
from astropy.coordinates import SkyCoord

from reproject import reproject_exact, reproject_interp

from AstroColour.AstroColour import RGB

import matplotlib.patheffects as PathEffects

%matplotlib widget

In [ ]:
plt.rc('text', usetex=True)
plt.rc('font', family='serif')
fig_width_pt = 244.0  # Get this from LaTeX using \the\columnwidth
text_width_pt = 508.0 # Get this from LaTeX using \the\textwidth

inches_per_pt = 1.0/72.27               # Convert pt to inches
golden_mean = (np.sqrt(5)-1.0)/2.0         # Aesthetic ratio
fig_width = fig_width_pt*inches_per_pt*1.5 # width in inches
fig_width_full = text_width_pt*inches_per_pt*1.5  # 17
fig_height =fig_width*golden_mean # height in inches
fig_size = [fig_width,fig_height] #(9,5.5) #(9, 4.5)
fig_height_full = fig_width_full*golden_mean

In [ ]:
path = '/Users/zgl12/Downloads/MAST_2026-04-09T0019/MAST_2026-04-09T0019/JWST/'
name = 'jw02107-o003_t003_miri_f770w_i2d.fits'

hdu = fits.open(path+name)
true_data = hdu[1].data
header = hdu[1].header
true_hdr = hdu[0].header
true_wcs = WCS(header)
# hdu.info()
# print(repr(header))
hdu.close()

In [ ]:

colour_cube = []
colour_cube.append(true_data)

for filt in [1000, 1130, 2100]:
    
    file_name = f'jw02107-o003_t003_miri_f{filt}w_i2d.fits'
    
    hdu = fits.open(path+file_name)
    temp_data = hdu[1].data
    header = hdu[1].header
    hdr = hdu[0].header
    wcs = WCS(header)
    
    # init_colour_cube = [red, green, blue]

    data, _ = reproject_interp((temp_data, wcs), true_wcs)
    colour_cube.append(temp_data)
    
    plt.figure()
    plt.imshow(temp_data, origin='lower', cmap='gray', vmin=np.nanpercentile(temp_data, 1), vmax=np.nanpercentile(temp_data, 99))
    plt.show()

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111)

# Plot the FITS image
ax.imshow(data, origin='lower', cmap='gray', 
          vmin=np.nanpercentile(data, 35),
          vmax=np.nanpercentile(data, 99.5))


x, y = wcs.all_world2pix(53.3948617, -36.1431472, 0)

plt.scatter(x, y, color = 'r')

# Overlay the DS9 regions
# for region in regions1:
#     pix_region = region.to_pixel(wcs)
#     pix_region.plot(ax=ax, color='red', linewidth=2)

# for region in regions2:
#     pix_region = region.to_pixel(wcs)
#     pix_region.plot(ax=ax, color='red', linewidth=1)

ax.set_xlabel("RA")
ax.set_ylabel("Dec")
plt.show()

In [ ]:
rgb = RGB(colour_cube,
          save = False, save_name = 'sn1983V', save_folder = '/Users/zgl12/', 
          epsf_plot=False, epsf = False,
          bkg_plot = False, temp_save = True, run = False, manual_override=0)

In [ ]:
len(colour_cube)

In [ ]:
val = 1.3

colour = rgb.master_plot(colour_cube, 
                         colours = ['rebeccapurple', 'blue', 'green', 'red'],
                         intensities = [0.1*val, 0.5*val, 1*val, 0.56*val], 
                         gamma = [1.5, 1.8, 1.5, 1.5],
                         norms = ['asinh', 'asinh', 'asinh', 'asinh'], 
                         uppers = [95, 95, 95, 95],
                         lowers = [10, 30, 10, 70], 
                         interactive=False)

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection = true_wcs)

# Plot the FITS image
ax.imshow(colour, origin='lower')


x, y = wcs.all_world2pix(53.38192, -36.14858, 0)

plt.scatter(x, y, color = 'r')

ax.set_xlim(x-150, x+150)
ax.set_ylim(y-150, y+150)

# Overlay the DS9 regions
# for region in regions1:
#     pix_region = region.to_pixel(wcs)
#     pix_region.plot(ax=ax, color='red', linewidth=2)

# for region in regions2:
#     pix_region = region.to_pixel(wcs)
#     pix_region.plot(ax=ax, color='red', linewidth=1)

ax.set_xlabel("Dec")
ax.set_ylabel("RA")
plt.savefig('colour_code_test.png', bbox_inches = 'tight', dpi = 600)
plt.show()

In [ ]:
np.save('/Users/zgl12/sneos_rgb.npy', colour)

In [ ]:
colour = np.load('/Users/zgl12/sneos_rgb.npy')

In [ ]:
colour.shape